# CrOSSD

## Jupyterlite stuff

In [ ]:
from pyodide.http import pyxhr
#cors = "https://corsproxy.io/?url="
#cors = "https://api.allorigins.win/get?url="
cors = "https://cors-anywhere.com/"

## Website Example

In [ ]:
from IPython.display import IFrame, display

display(IFrame("https://health.crossd.tech", 1050,700))

## API Endpoints

### `/api/projects`
Get a list of all projects

In [ ]:
#import requests
import json
from rich import print

project = "lorabridge/lorabridge"

# response = requests.post('https://health.crossd.tech/api/projects')
# Synchronous HTTP request
response = pyxhr.post(cors + "https://health.crossd.tech/api/projects")
data = response.json()
#data = json.loads(data['contents'])

In [ ]:
print(f"Project count: {len(data)}\n")

In [ ]:
print(json.dumps(data,indent=4))

### `/api/snapshots`
Get a list of snapshots (timestamps of the particular retrievals) of the specified project

In [ ]:
headers = {
    'Content-Type': 'application/json',
}

json_data = {
    'term': project,
}

#response = requests.post('https://health.crossd.tech/api/snapshots', headers=headers, json=json_data)
response = pyxhr.post(cors + 'https://health.crossd.tech/api/snapshots', headers=headers, json=json_data)
data = response.json()

In [ ]:
print(f"Snapshot for project {json_data['term']}")
print(f"Snapshot count: {len(data)}")

In [ ]:
print(json.dumps(data,indent=4))

### `/api/metrics`
Get the calculated metrics for the specified project at the desired point in time (snapshot)

In [ ]:
json_data = {
    'term': project,
    'timestamp': data[-1],
}

# response = requests.post('https://health.crossd.tech/api/metrics', headers=headers, json=json_data)
response = pyxhr.post(cors + 'https://health.crossd.tech/api/metrics', headers=headers, json=json_data)
data = response.json()

In [ ]:
print(json.dumps(data, indent = 4))

### `/api/metrics/avg`
Get average of most important metrics

In [ ]:
# response = requests.post('https://health.crossd.tech/api/metrics/avg', headers=headers)
response = pyxhr.post(cors + 'https://health.crossd.tech/api/metrics/avg', headers=headers)
data = response.json()

In [ ]:
print(json.dumps(data, indent = 4))

### `/api/repo`
Get the retrieved raw data of the repository at the specified point in time (snapshot)

In [ ]:
json_data = {
    'term': project,
    'timestamp': json_data['timestamp'],
}

# response = requests.post('https://health.crossd.tech/api/repo', headers=headers, json=json_data)
response = pyxhr.post(cors + 'https://health.crossd.tech/api/repo', headers=headers, json=json_data)
data = response.json()

In [ ]:
print(json.dumps(data, indent = 4))

## Pipenv dependency example

### Pipfile Content

In [ ]:
from rich.markup import escape
print(escape(open("Pipfile").read()))

### Retrieve Github URIs from Pipfile dependencies

In [ ]:
import tomllib
import re

# read dependencies
packages = tomllib.loads(open("Pipfile").read())["packages"].keys()
projects=[]

for pkg in packages:
    # get pypi info for each package
    #response = requests.get(f"https://pypi.org/pypi/{pkg}/json")
    response = pyxhr.get(f"https://pypi.org/pypi/{pkg}/json")
    data = response.json()
    # Check if there are any github URIs
    for name, url in data["info"]["project_urls"].items():
        if m:=re.match(r"https://github\.com/([a-zA-Z0-9-_]+)/([a-zA-Z0-9-_]+)",url):
            projects.append("/".join(m.group(1,2)))
            break
print(projects)


### Retrieve snapshots for the projects in Pipfile

In [ ]:
snapshots={}

for project in projects:
    json_data = {
    'term': project,
    }
    
    #response = requests.post('https://health.crossd.tech/api/snapshots', headers=headers, json=json_data)
    response = pyxhr.post(cors + 'https://health.crossd.tech/api/snapshots', headers=headers, json=json_data)
    data = response.json()
    snapshots[project] = data

print(snapshots)

### Retrieve metrics for latest snapshot of the projects

In [ ]:
metrics = {}

for project in snapshots:
    if snapshots[project]:
        for snap in reversed(snapshots[project]):
            json_data = {
                'term': project,
                'timestamp': snap,
            }
    
            #response = requests.post('https://health.crossd.tech/api/metrics', headers=headers, json=json_data)
            response = pyxhr.post(cors + 'https://health.crossd.tech/api/metrics', headers=headers, json=json_data)
            data = response.json()
            metrics[project] = data
            if data["metrics"]:
                break

In [ ]:
print(metrics)

### Compare to Average

In [ ]:
# retrieve average values of metrics 

# response = requests.post('https://health.crossd.tech/api/metrics/avg', headers=headers)
response = pyxhr.post(cors + 'https://health.crossd.tech/api/metrics/avg', headers=headers)
data = response.json()

elephant = data["avg"]["elephant_factor"]
maturity = data["avg"]["maturity_level"]
crit = data["avg"]["criticality_score"]
support_rate = data["avg"]["support_rate"]
github_community_health_percentage = data["avg"]["github_community_health_percentage"]

print("Averages:\n")
print(f"Elephant factor: \t\t\t{elephant}")
print(f"Maturity level: \t\t\t{maturity}")
print(f"Criticality score: \t\t\t{crit}")
print(f"Rupport Rate: \t\t\t\t{support_rate}")
print(f"Github community health percentage: \t{github_community_health_percentage}")

In [ ]:
def is_within_range_pos(value: float, reference: float, threshold: float) -> bool:
    return value > reference or abs(value - reference) <= reference * threshold

def print_color(reference, value):
    if is_within_range_pos(value, reference, threshold = 0.15):
        print("\033[32;1m"+str(value)+"\033[00m")
    elif is_within_range_pos(value, reference, threshold = 0.25):
        print("\033[33;1m"+str(value)+"\033[00m")
    else:
        print("\033[91;1m"+str(value)+"\033[00m")

In [ ]:
for project in projects:
    print("\033[1m" + project + "\033[00m")
    
    print("Elephant factor: ",end="\t\t\t")
    print_color(elephant, metrics[project]["metrics"]["elephant_factor"])
    print("Maturity level: ",end="\t\t\t")
    print_color(maturity, metrics[project]["metrics"]["maturity_level"])
    print("Criticality score: ",end="\t\t\t")
    print_color(crit, metrics[project]["metrics"]["criticality_score"])
    print("Support rate: ",end="\t\t\t\t")
    print_color(support_rate, metrics[project]["metrics"]["support_rate"])
    print("Github community health percentage: ",end="\t")
    print_color(github_community_health_percentage, metrics[project]["metrics"]["github_community_health_percentage"]["custom_health_score"])
    print()